# Active modulation of zonal flows in modified Hasegawa--Wakatani turbulence
### Reproduction notebook

This notebook reproduces all five figures of the accompanying report. Run the cells
top to bottom (`Runtime -> Run all`). Cell 1 defines the solver once; every cell
below only calls it.

- **Cell 1** - setup, spectral solver, and `run_sim(F0, omega_AC, seed)`
- **Cell 2** - Figures 1 & 2: baseline and forced field snapshots
- **Cell 3** - Figures 3 & 4: F0 sweep (time-averaged vs single-snapshot)
- **Cell 4** - Figure 5: 2D operational window over (F0, omega_AC)
- **Cell 5** - statistical significance (five-seed ensembles)

Requirements: `numpy`, `matplotlib` (preinstalled on Google Colab).
Random seeds are fixed, so the numbers below are reproducible run to run.

*Code written with AI assistance (Anthropic's Claude); all simulations, verification
choices, and interpretation are the author's own.*

In [ ]:
# ============================================================
# Cell 1 - setup, solver, and run_sim()  (defined ONCE)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# ---- grid & wavenumbers ----
N = 64
L = 20*np.pi
x = np.linspace(0, L, N, endpoint=False)
y = np.linspace(0, L, N, endpoint=False)
X, Y = np.meshgrid(x, y)
k = np.fft.fftfreq(N, d=L/N)*2*np.pi
kx, ky = np.meshgrid(k, k)
k2 = kx**2 + ky**2
k2_inv = 1.0/np.where(k2 == 0, 1, k2)        # 1/k^2 with k=0 avoided

# ---- physical constants (paper values) ----
alpha = 0.5      # adiabaticity (drift-wave drive)
kappa = 1.0      # background density gradient
nu    = 2e-4     # hyper-diffusion coefficient
dt    = 0.02     # time step
k_zf  = 4*2*np.pi/L                 # zonal forcing wavenumber
forcing_shape = np.cos(k_zf*X)      # x-dependent stripe shape of F_AC

# ---- dealiasing mask (2/3 rule) ----
kmax = np.max(np.abs(k))
mask = (np.abs(kx) < 2/3*kmax) & (np.abs(ky) < 2/3*kmax)

# ---- spectral building blocks ----
def ddx(f): return np.real(np.fft.ifft2(1j*kx*np.fft.fft2(f)))
def ddy(f): return np.real(np.fft.ifft2(1j*ky*np.fft.fft2(f)))
def get_phi(w):
    ph = -np.fft.fft2(w)*k2_inv; ph[0, 0] = 0
    return np.real(np.fft.ifft2(ph))
def poisson(f, g): return ddx(f)*ddy(g) - ddy(f)*ddx(g)
def tilde(f): return f - f.mean(axis=0, keepdims=True)   # remove zonal average

def run_sim(F0, omega_AC=0.1, seed=42, nsteps=3000, return_history=False,
            return_fields=False):
    """Integrate the modified Hasegawa-Wakatani system with an external
    zonal drive F_AC = F0*cos(omega_AC*t)*cos(k_zf*x).

    Returns time-averaged (zonal_energy, turbulent_energy) over the final 30%
    of the run. If the run diverges, returns (nan, nan).
    Set return_history=True to also get (times, EZF_hist, Eturb_hist) for plotting,
    and return_fields=True to also get the final (n, phi) fields.
    """
    np.random.seed(seed)
    n = 0.01*np.random.randn(N, N)
    w = 0.01*np.random.randn(N, N)

    def nonlinear(n, w, t):
        phi = get_phi(w)
        coupling = alpha*(tilde(phi) - tilde(n))
        F_AC = F0*np.cos(omega_AC*t)*forcing_shape
        dn = -poisson(phi, n) + coupling - kappa*ddy(phi)
        dw = -poisson(phi, w) + coupling + F_AC
        return dn, dw

    times, EZF_hist, Eturb_hist = [], [], []
    EZF_s, Eturb_s = [], []                       # final-30% samples for averaging
    for step in range(nsteps):
        t = step*dt
        dn1, dw1 = nonlinear(n, w, t)
        nm = n + 0.5*dt*dn1; wm = w + 0.5*dt*dw1
        dn2, dw2 = nonlinear(nm, wm, t + 0.5*dt)
        n = n + dt*dn2; w = w + dt*dw2
        nh = np.fft.fft2(n)*mask*np.exp(-nu*k2**2*dt)
        wh = np.fft.fft2(w)*mask*np.exp(-nu*k2**2*dt)
        n = np.real(np.fft.ifft2(nh)); w = np.real(np.fft.ifft2(wh))
        if not np.isfinite(n).all():              # diverged
            if return_history or return_fields:
                return np.nan, np.nan, times, EZF_hist, Eturb_hist, n, get_phi(w)
            return np.nan, np.nan
        if step % 50 == 0:                        # history for time traces
            phi = get_phi(w); ph = np.fft.fft2(phi)
            E = 0.5*k2*np.abs(ph)**2/N**4
            ezf = E[:, 0].sum()
            times.append(t); EZF_hist.append(ezf); Eturb_hist.append(E.sum() - ezf)
        if step > 0.7*nsteps:                     # final-30% samples
            phi = get_phi(w); ph = np.fft.fft2(phi)
            E = 0.5*k2*np.abs(ph)**2/N**4
            EZF_s.append(E[:, 0].sum()); Eturb_s.append(E.sum() - E[:, 0].sum())

    EZF, Eturb = np.mean(EZF_s), np.mean(Eturb_s)
    if return_history or return_fields:
        return EZF, Eturb, times, EZF_hist, Eturb_hist, n, get_phi(w)
    return EZF, Eturb

print("Cell 1 ready: run_sim() defined. Grid %dx%d, L=%.2f." % (N, N, L))

In [ ]:
# ============================================================
# Cell 2 - Figures 1 & 2: baseline (F0=0) and forced (F0=0.05) fields
# ============================================================
NSTEPS_FIELD = 4000

def field_figure(F0, fname, title_tag):
    _, _, times, EZF_h, Eturb_h, n, phi = run_sim(
        F0, omega_AC=0.1, seed=42, nsteps=NSTEPS_FIELD, return_fields=True)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    im0 = ax[0].pcolormesh(X, Y, n, cmap='RdBu_r', shading='auto')
    ax[0].set_title('density n (%s)' % title_tag); ax[0].axis('square')
    fig.colorbar(im0, ax=ax[0])
    im1 = ax[1].pcolormesh(X, Y, phi, cmap='RdBu_r', shading='auto')
    ax[1].set_title('phi potential (%s)' % title_tag); ax[1].axis('square')
    fig.colorbar(im1, ax=ax[1])
    ax[2].plot(times, EZF_h, 'g-', label='zonal flow energy')
    ax[2].plot(times, Eturb_h, 'r--', label='turbulence energy')
    ax[2].set_xlabel('time'); ax[2].set_yscale('log'); ax[2].legend()
    ax[2].set_title('energy vs time')
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print("saved", fname)

field_figure(0.0,  'Figure_1.png', 'baseline, no forcing')   # Fig 1
field_figure(0.05, 'Figure_2.png', 'with F_AC, F0=0.05')     # Fig 2

In [ ]:
# ============================================================
# Cell 3 - Figures 3 & 4: F0 sweep, time-averaged vs single-snapshot
# ============================================================
F0_list = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08]

# --- time-averaged sweep (Figure 3, the correct diagnostic) ---
EZF_avg, Eturb_avg = [], []
for F0 in F0_list:
    ezf, eturb = run_sim(F0, omega_AC=0.1, seed=42, nsteps=3000)
    EZF_avg.append(ezf); Eturb_avg.append(eturb)
    print("F0=%.2f -> zonal=%.2e, turb=%.2e" % (F0, ezf, eturb))

plt.figure(figsize=(7, 5))
plt.plot(F0_list, EZF_avg, 'go-', label='zonal flow energy')
plt.plot(F0_list, Eturb_avg, 'rs--', label='turbulence energy')
plt.xlabel('pump strength F0'); plt.ylabel('energy'); plt.yscale('log')
plt.legend(); plt.grid(alpha=0.3)
plt.title('parameter sweep (time-averaged): does turbulence fall as F0 rises?')
plt.savefig('Figure_3.png', dpi=150, bbox_inches='tight')
plt.show()

# --- single-snapshot sweep (Figure 4, kept as honest before/after) ---
def run_sim_snapshot(F0, omega_AC=0.1, seed=42, nsteps=3000):
    """Identical integration but measures energy only at the FINAL instant."""
    np.random.seed(seed)
    n = 0.01*np.random.randn(N, N); w = 0.01*np.random.randn(N, N)
    def nl(n, w, t):
        phi = get_phi(w); coupling = alpha*(tilde(phi) - tilde(n))
        F_AC = F0*np.cos(omega_AC*t)*forcing_shape
        return (-poisson(phi, n) + coupling - kappa*ddy(phi),
                -poisson(phi, w) + coupling + F_AC)
    for step in range(nsteps):
        t = step*dt
        dn1, dw1 = nl(n, w, t); nm = n+0.5*dt*dn1; wm = w+0.5*dt*dw1
        dn2, dw2 = nl(nm, wm, t+0.5*dt); n = n+dt*dn2; w = w+dt*dw2
        nh = np.fft.fft2(n)*mask*np.exp(-nu*k2**2*dt)
        wh = np.fft.fft2(w)*mask*np.exp(-nu*k2**2*dt)
        n = np.real(np.fft.ifft2(nh)); w = np.real(np.fft.ifft2(wh))
        if not np.isfinite(n).all(): return np.nan, np.nan
    phi = get_phi(w); ph = np.fft.fft2(phi)
    E = 0.5*k2*np.abs(ph)**2/N**4
    return E[:, 0].sum(), E.sum() - E[:, 0].sum()

EZF_snap, Eturb_snap = [], []
for F0 in F0_list:
    ezf, eturb = run_sim_snapshot(F0)
    EZF_snap.append(ezf); Eturb_snap.append(eturb)

plt.figure(figsize=(7, 5))
plt.plot(F0_list, EZF_snap, 'go-', label='zonal flow energy')
plt.plot(F0_list, Eturb_snap, 'rs--', label='turbulence energy')
plt.xlabel('pump strength F0'); plt.ylabel('energy'); plt.yscale('log')
plt.legend(); plt.grid(alpha=0.3)
plt.title('parameter sweep (single snapshot): the misleading version')
plt.savefig('Figure_4.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figs 3 & 4 saved. Note the inflated baseline and hidden minimum in Fig 4.")

In [ ]:
# ============================================================
# Cell 4 - Figure 5: 2D operational window over (F0, omega_AC)
# ============================================================
F0_list    = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08]
omega_list = [0.02, 0.05, 0.1, 0.2, 0.4, 0.8]

turb_grid = np.full((len(omega_list), len(F0_list)), np.nan)
for i, omega_AC in enumerate(omega_list):
    for j, F0 in enumerate(F0_list):
        _, eturb = run_sim(F0, omega_AC=omega_AC, seed=42, nsteps=3000)
        turb_grid[i, j] = eturb
        print("omega=%.2f, F0=%.3f -> turb=%.2e" % (omega_AC, F0, eturb))
    print("-"*40)

baseline = turb_grid[:, 0].mean()              # F0=0 column = natural turbulence
diverged = turb_grid > 3*baseline              # diagnostic cutoff (not a physical boundary)
plot_grid = np.where(diverged, np.nan, turb_grid)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(plot_grid, origin='lower', aspect='auto', cmap='viridis_r',
               norm=LogNorm(vmin=np.nanmin(plot_grid), vmax=np.nanmax(plot_grid)))
for i in range(len(omega_list)):
    for j in range(len(F0_list)):
        if diverged[i, j]:
            ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                         fill=True, facecolor='red', alpha=0.3, hatch='xx'))
ax.set_xticks(range(len(F0_list)));    ax.set_xticklabels(F0_list)
ax.set_yticks(range(len(omega_list))); ax.set_yticklabels(omega_list)
ax.set_xlabel('pump strength F0'); ax.set_ylabel('drive frequency omega_AC')
ax.set_title('operational window: turbulence energy over (F0, omega_AC)')
cbar = fig.colorbar(im, ax=ax); cbar.set_label('turbulence energy (log)')
plt.tight_layout()
plt.savefig('Figure_5.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 5 saved. baseline=%.3f, divergence cutoff=%.3f" % (baseline, 3*baseline))

In [ ]:
# ============================================================
# Cell 5 - statistical significance (five-seed ensembles)
# ============================================================
seeds = [1, 2, 3, 4, 5]

# --- Table 1: baseline (F0=0) vs optimal (F0=0.02) at omega_AC=0.1 ---
print("Table 1: F0=0 vs F0=0.02 at omega_AC=0.1")
print("F0     mean      std       rel.spread")
for F0 in [0.0, 0.02]:
    vals = np.array([run_sim(F0, omega_AC=0.1, seed=s, nsteps=3000)[1] for s in seeds])
    print("%.2f   %.4f   %.4f   %.1f%%" % (F0, vals.mean(), vals.std(),
                                           100*vals.std()/vals.mean()))

# --- optimal-frequency check: omega 0.05 vs 0.06 vs 0.07 at F0=0.02 ---
print("\nOptimal-frequency check at F0=0.02")
print("omega  mean      std       rel.spread")
for omega_AC in [0.05, 0.06, 0.07]:
    vals = np.array([run_sim(0.02, omega_AC=omega_AC, seed=s, nsteps=3000)[1] for s in seeds])
    print("%.2f   %.4f   %.4f   %.1f%%" % (omega_AC, vals.mean(), vals.std(),
                                           100*vals.std()/vals.mean()))